# Допуск: GEPA: Reflective Prompt Evolution Can Outperform Reinforcement Learning

**Статья:** L. A. Agrawal и др. — <https://arxiv.org/abs/2507.19457v2>.
Задания сверены с версией v2 от 14.02.2026 (ICLR 2026).

5 заданий. Под заголовком каждого задания указан его тип:

- **вопрос по коду** — в коде нет отклонений: ответьте на вопросы через код и статью;
- **расхождение со статьёй** — найдите одно отклонение от описанного авторами метода;
- **неверный комментарий** — код корректен, но один комментарий утверждает неправду.

Для отклонения укажите место, объясните, что не так, сошлитесь на раздел, алгоритм,
рисунок или таблицу статьи и опишите исправление словами. Писать, запускать и отлаживать
код не нужно. Вопросы относятся только к заданиям без отклонений.

Общие импорты:

In [ ]:
from copy import deepcopy
import torch

Система `system` содержит словарь промптов `system.prompts`, модели с весами Θ и логику
вызовов C. `run_batch(system, batch, capture_trace=False)` выполняет систему по одному разу
на каждый пример; возвращает `scores` — тензор `(B,)`, `outputs` и, при запросе, `traces`.
Функции, которые используются, но не показаны, считайте корректными. Примеры — сокращённые
модели механизмов статьи, а не полная библиотека GEPA.

## Задание 1. Одно отражение над траекториями

**Тип задания:** вопрос по коду — в коде нет отклонений, ответьте на вопросы.

Шаг Reflective Prompt Mutation (§3, алгоритм 1, прил. C). Модуль `module` уже выбран;
в этом примере он вызывается один раз в каждой траектории. `feedback_fn` реализует μ_f:
оценивает готовые результат и траекторию, возвращая оценку и текст, без нового запуска
системы. `reflect` — один вызов reflection LM, предлагающей новый промпт.

In [ ]:
def mutate(parent, module, minibatch, feedback_fn, reflect):
    before = run_batch(parent, minibatch, capture_trace=True)
    records = []
    for item, output, trace in zip(minibatch, before.outputs, before.traces):
        score, text = feedback_fn(item, output, trace)
        records.append({"module_trace": trace[module],
                        "score": score, "feedback": text})
    child = deepcopy(parent)
    child.prompts[module] = reflect(parent.prompts[module], records)
    after = run_batch(child, minibatch)
    if after.scores.mean() > before.scores.mean():
        return child
    return None

**Вопросы.**

1. Что меняется в параметрах ⟨Π, Θ⟩ и что происходит с C? Чем это отличается от GRPO и
   почему такой способ применим к закрытой модели (§2–3, прил. E.4)?
2. Какие два источника диагностической информации попадают в `records`? Что теряется,
   если оставить только скалярные оценки? Приведите пример из описания feedback для
   HotpotQA или генерации кода (§3, прил. E.1).
3. При размере minibatch 3 сколько здесь запусков всей системы и сколько вызовов `reflect`?
   Почему число rollouts не равно числу всех LLM-вызовов (§2, прил. N)?

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 2. Откуда продолжать поиск

**Тип задания:** расхождение со статьёй — код реализует не то, что описано в статье.

Выбор родителя для следующей мутации. `scores` — оценки кандидатов на D_pareto, тензор
`(K, N)`: строки — кандидаты, столбцы — отдельные примеры. Найдите одно отклонение от
основного метода GEPA (§3.1, алгоритм 2, табл. 3).

In [ ]:
def select_candidate(pool, scores):
    average = scores.mean(dim=1)                         # (K,)
    index = int(average.argmax())
    return pool[index]

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 3. Совместить две ветви

**Тип задания:** расхождение со статьёй — код реализует не то, что описано в статье.

Сборка потомка в System-Aware Merge (прил. D.1, алгоритмы 3–4). `ancestor` — общий
предок; пара `left`, `right` уже прошла проверки допустимости merge. В данном примере
каждый модуль изменён не более чем в одной ветви, поэтому конфликтов правок нет.
`left_score`, `right_score` — средние оценки родителей. Найдите одно отклонение.

In [ ]:
def merge(ancestor, left, right, left_score, right_score):
    child = deepcopy(ancestor)
    source = left if left_score > right_score else right
    for module in child.prompts:
        child.prompts[module] = source.prompts[module]
    return child

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 4. Последний выбор

**Тип задания:** расхождение со статьёй — код реализует не то, что описано в статье.

Бюджет оптимизации закончился. `evaluate(system, dataset)` возвращает среднюю оценку
системы на наборе; `d_pareto` — validation, `d_test` — отложенный test. Найдите одно
отклонение от протокола статьи (§4, алгоритм 1, строка 21; прил. E.4).

In [ ]:
def choose_final(pool, d_pareto, d_test, evaluate):
    scores = torch.tensor([evaluate(system, d_test) for system in pool])
    winner = int(scores.argmax())
    return pool[winner], float(scores[winner])

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 5. Что даёт crossover

**Тип задания:** неверный комментарий — код корректен, но один комментарий утверждает неправду.

Результаты GEPA и GEPA+Merge из табл. 1–2. Строки — Qwen3 8B и GPT-4.1 Mini;
столбцы — HotpotQA, IFBench, HoVer, PUPA, AIME-2025, LiveBench-Math. Разности выражены
в процентных пунктах. Найдите один неверный комментарий (§4, Observation 5).

In [ ]:
def merge_gains():
    gepa = torch.tensor([[62.33, 38.61, 52.33, 91.85, 32.00, 51.95],
                         [69.00, 52.72, 51.67, 94.47, 59.33, 64.13]])
    merged = torch.tensor([[64.33, 28.23, 51.67, 86.26, 32.00, 51.95],
                           [65.67, 55.95, 56.67, 96.46, 59.33, 64.13]])
    gains = merged - gepa                               # (2, 6), percentage points
    # Crossover improves GEPA on every benchmark for both models, independently of allocation.
    return gains, gains.mean(dim=1)

**Ваш ответ:**

_Напишите ответ здесь._